# Cross-Platform Community Evolution: Step 2 Data Cleaning & Preparation

**Project**: Show Reel Media Group Community Evolution  
**Platforms**: Instagram · TikTok · YouTube  
**Objective**: Unified Python processing pipeline with three optimised export formats

## Output (written to `gs://afb_showreel/Preped_Comments/`)
- **comments_llm.jsonl** — raw text + metadata for LLM / CAG input
- **comments_ml.parquet** — flattened columnar features for XGBoost / LightGBM
- **comments_gml.parquet** — edge-list format for PyTorch Geometric GNNs

## Section 1: Imports and Configuration

In [ ]:
!pip install -q emoji gcsfs pyarrow pandas scipy

In [ ]:
# Colab Enterprise runs as a service account — Application Default Credentials
# are resolved automatically.  No interactive auth prompt needed or allowed.

In [ ]:
import os

# ── GCP configuration ─────────────────────────────────────────────────────────
GCP_PROJECT_ID = 'gen-lang-client-0792749758'
GCP_BUCKET     = 'afb_showreel'
GCP_LOCATION   = 'us-central1'

# Set project hints for all Google Cloud client libraries
os.environ['GOOGLE_CLOUD_PROJECT']       = GCP_PROJECT_ID
os.environ['GOOGLE_CLOUD_QUOTA_PROJECT'] = GCP_PROJECT_ID

# Derived path roots
GCS_BUCKET = GCP_BUCKET   # alias kept for legacy references
gcs        = f'gs://{GCP_BUCKET}'

In [ ]:
import json
import logging
import time
from typing import Dict, List, Tuple, Any, Optional
from dataclasses import dataclass, asdict
from enum import Enum
from pathlib import Path
import re
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime, timezone
import emoji
from scipy.stats import entropy as scipy_entropy

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

## Section 2: Type Definitions and Emoji Taxonomy

In [ ]:
class PlatformType(Enum):
    INSTAGRAM = "instagram"
    TIKTOK = "tiktok"
    YOUTUBE = "youtube"

class EmojiCategory(Enum):
    LOVE = "love"
    CELEBRATION = "celebration"
    HUMOR = "humor"
    INQUIRY = "inquiry"
    OTHER = "other"

@dataclass
class RawComment:
    """Unified raw comment structure"""
    platform: str
    author_id: str
    media_id: str
    comment_text: str
    reply_to_comment_id: Optional[str]
    timestamp: str
    original_from_id: Optional[str] = None
    original_uid: Optional[str] = None

@dataclass
class ProcessedCommentML:
    """ML-optimized columnar representation"""
    comment_id: str
    author_id: str
    media_id: str
    platform: str
    text_length: int
    word_count: int
    emoji_count: int
    unique_emoji_count: float
    emoji_entropy: float
    emoji_variety_ratio: float
    emoji_per_word_ratio: float
    url_count: int
    mention_count: int
    hashtag_count: int
    exclamation_count: int
    question_count: int
    avg_word_length: float
    has_numbers: int
    has_links: int
    timestamp: str

@dataclass
class ProcessedCommentGML:
    """Graph ML edge-list representation"""
    comment_id: str
    author_id: str
    media_id: str
    reply_to_comment_id: Optional[str]
    platform: str
    timestamp: str

# Emoji Taxonomy Mapping
EMOJI_TAXONOMY = {
    EmojiCategory.LOVE: ['❤️', '💕', '💖', '😍', '🥰', '💗', '💝', '💞', '👍', '💯'],
    EmojiCategory.CELEBRATION: ['🎉', '🎊', '🥳', '🎈', '✨', '⭐', '🌟', '🔥', '👏', '🙌'],
    EmojiCategory.HUMOR: ['😂', '🤣', '😄', '😁', '😆', '🤪', '😜', '🤩', '😏', '💀'],
    EmojiCategory.INQUIRY: ['❓', '❔', '🤔', '🧐', '😕', '🤨', '💭', '🙄', '😐', '😒']
}

# Reverse mapping for fast lookup
EMOJI_TO_CATEGORY = {}
for category, emojis in EMOJI_TAXONOMY.items():
    for e in emojis:
        EMOJI_TO_CATEGORY[e] = category.value

## Section 3: Utility Functions (Emoji Extraction & Feature Engineering)

In [ ]:
def compute_shannon_entropy(emoji_list: List[str]) -> float:
    """Compute Shannon Entropy of emoji distribution.

    High entropy = diverse emoji usage; Low entropy = repetitive usage.
    Returns 0 if no emojis present.
    """
    if not emoji_list:
        return 0.0

    counts = Counter(emoji_list)
    frequencies = np.array(list(counts.values())) / len(emoji_list)
    return float(scipy_entropy(frequencies))

def extract_emojis_safe(text: str) -> List[str]:
    """Extract emojis using emoji library to preserve ZWJ sequences.

    Properly handles composite sequences like skin tone modifiers,
    gender variations, and coupled glyphs.
    """
    try:
        return [e['emoji'] for e in emoji.emoji_list(text)]
    except Exception as e:
        logger.warning(f"Emoji extraction failed: {e}")
        return []

def categorize_emojis(emoji_list: List[str]) -> Dict[str, int]:
    """Categorize extracted emojis into semantic buckets."""
    categories = defaultdict(int)
    for em in emoji_list:
        category = EMOJI_TO_CATEGORY.get(em, EmojiCategory.OTHER.value)
        categories[category] += 1
    return dict(categories)

def compute_emoji_features(emoji_list: List[str], word_count: int) -> Tuple[float, float, float]:
    """Compute emoji features: variety_ratio, entropy, per_word_ratio."""
    if not emoji_list:
        return 0.0, 0.0, 0.0

    unique_count = len(set(emoji_list))
    variety_ratio = unique_count / len(emoji_list) if emoji_list else 0.0
    entropy_val = compute_shannon_entropy(emoji_list)
    per_word_ratio = len(emoji_list) / word_count if word_count > 0 else 0.0

    return variety_ratio, entropy_val, per_word_ratio

## Section 4: Regex Pre-compilation & Advanced Text Preprocessing

In [ ]:
# Pre-compile all regex patterns for linear-time O(n) batch processing
REGEX_PATTERNS = {
    'url': re.compile(r'https?://\S+|www\.\S+'),
    'mention': re.compile(r'@\w+'),
    'hashtag': re.compile(r'#\w+'),
    'exclamation': re.compile(r'!'),
    'question': re.compile(r'\?'),
    'numbers': re.compile(r'\d'),
    'whitespace': re.compile(r'\s+')
}

def advanced_text_preprocessing(text: str) -> Dict[str, Any]:
    """Advanced preprocessing with linguistic feature extraction.

    Returns dict with counts and derived metrics for ML features.
    """
    if not isinstance(text, str) or not text.strip():
        return {
            'text_length': 0,
            'word_count': 0,
            'emoji_count': 0,
            'unique_emoji_count': 0,
            'emoji_entropy': 0.0,
            'emoji_variety_ratio': 0.0,
            'emoji_per_word_ratio': 0.0,
            'url_count': 0,
            'mention_count': 0,
            'hashtag_count': 0,
            'exclamation_count': 0,
            'question_count': 0,
            'avg_word_length': 0.0,
            'has_numbers': 0,
            'has_links': 0
        }

    text = text.strip()
    text_length = len(text)

    # Extract emojis using safe method
    emojis = extract_emojis_safe(text)
    emoji_count = len(emojis)
    unique_emoji_count = len(set(emojis))

    # Remove emojis for word-level analysis
    text_without_emoji = ''.join(c for c in text if c not in emojis)

    # Word tokenization
    words = REGEX_PATTERNS['whitespace'].split(text_without_emoji.strip())
    words = [w for w in words if w]  # Remove empty strings
    word_count = len(words)

    # Compute emoji features
    variety_ratio, entropy_val, per_word_ratio = compute_emoji_features(emojis, word_count)

    # Pattern matching with pre-compiled regex
    url_count = len(REGEX_PATTERNS['url'].findall(text))
    mention_count = len(REGEX_PATTERNS['mention'].findall(text))
    hashtag_count = len(REGEX_PATTERNS['hashtag'].findall(text))
    exclamation_count = len(REGEX_PATTERNS['exclamation'].findall(text))
    question_count = len(REGEX_PATTERNS['question'].findall(text))
    has_numbers = 1 if REGEX_PATTERNS['numbers'].search(text) else 0
    has_links = 1 if url_count > 0 else 0

    # Average word length
    avg_word_length = np.mean([len(w) for w in words]) if words else 0.0

    return {
        'text_length': text_length,
        'word_count': word_count,
        'emoji_count': emoji_count,
        'unique_emoji_count': unique_emoji_count,
        'emoji_entropy': entropy_val,
        'emoji_variety_ratio': variety_ratio,
        'emoji_per_word_ratio': per_word_ratio,
        'url_count': url_count,
        'mention_count': mention_count,
        'hashtag_count': hashtag_count,
        'exclamation_count': exclamation_count,
        'question_count': question_count,
        'avg_word_length': float(avg_word_length),
        'has_numbers': has_numbers,
        'has_links': has_links
    }

## Section 5: Unified Pipeline - Platform Normalization & Processing

In [ ]:

# ── Section 5: Hybrid Heterogeneous Graph Pipeline ────────────────────────────
#
# ARCHITECTURAL REDESIGN — Hybrid GML Infrastructure
# ───────────────────────────────────────────────────
# Replaces the legacy homogeneous `comments_gml.parquet` with a full
# heterogeneous graph schema that serves two downstream consumers:
#
#   1. Static Graph Analytics (cuGraph / iGraph)
#      Load the flat node/edge Parquet files, build a directed multigraph,
#      run Louvain community detection or PageRank.  Output metrics feed
#      back into the tabular ML model as graph-structural features.
#
#   2. Deep Representation Learning (PyTorch Geometric)
#      The same Parquet files load directly into a PyG HeteroData object
#      for link-prediction and cross-platform comment embedding tasks.
#
# Output (written to gs://{bucket}/HeteroGraph/):
#   nodes_author.parquet       nodes_comment.parquet     nodes_media.parquet
#   edges_posted.parquet       edges_belongs_to.parquet
#   edges_replies_to.parquet   edges_derived_from.parquet
#   edges_similar_to.parquet
#
# Platform topology rules:
#   Instagram  — POSTED + REPLIES_TO + BELONGS_TO
#   TikTok     — POSTED + BELONGS_TO +
#                virtual SIMILAR_TO (cosine sim on ML feature vectors,
#                grouped per video to bound O(n_per_video²) complexity)
#   YouTube    — POSTED + REPLIES_TO + BELONGS_TO +
#                DERIVED_FROM (Short → parent LongForm, injected via
#                yt_short_to_long mapping built from Short_to_Long_connection/)
#
# ML (ProcessedCommentML) and LLM extraction paths are UNCHANGED.

# ── Platform prefix registry ──────────────────────────────────────────────────

PLATFORM_PREFIX: Dict[str, str] = {
    'instagram': 'ig',
    'tiktok':    'tk',
    'youtube':   'yt',
}

# Feature columns used as the cosine-similarity proxy matrix for TikTok.
# Swap with a dense LLM topic-embedding array when R^K vectors are ready;
# the _compute_similar_to_edges() interface is identical.
_TIKTOK_SIM_FEATURES: List[str] = [
    'text_length', 'word_count', 'emoji_count', 'unique_emoji_count',
    'emoji_entropy', 'emoji_variety_ratio', 'emoji_per_word_ratio',
    'url_count', 'mention_count', 'hashtag_count',
    'exclamation_count', 'question_count', 'avg_word_length',
    'has_numbers', 'has_links',
]

# ── Heterogeneous Node schemas ────────────────────────────────────────────────

@dataclass
class NodeAuthor:
    """User / creator node.  One row per unique (platform, author) pair.

    canonical author_id = f"{prefix}_author_{native_uid}"
    e.g. 'ig_author_23948', 'yt_author_UCxxxxxx'
    """
    author_id: str
    platform:  str

@dataclass
class NodeComment:
    """Comment node with downstream embedding placeholder slots.

    Sentinel values for embedding columns signal 'not yet computed':
      sentiment_neg/neu/pos = 0.0  (Gemini sentiment phase fills these)
      topic_vec_dim = 0            (set by topic-modelling phase)

    The full K-dim topic vector is stored as a dense array in
    topic_embeddings_{platform}.npy, joined on comment_id at training time.

    canonical comment_id = f"{prefix}_comment_{native_cid}"
    """
    comment_id:    str
    author_id:     str
    media_id:      str
    platform:      str
    timestamp:     str
    sentiment_neg: float = 0.0   # R³ probability simplex – neg
    sentiment_neu: float = 0.0   #                         – neu
    sentiment_pos: float = 0.0   #                         – pos
    topic_vec_dim: int   = 0     # declared K (0 = not yet computed)

@dataclass
class NodeMedia:
    """Post / video node.

    YouTube-specific fields (parent_media_id, is_short, format_type) are
    None / False / 'video' for other platforms.

    canonical media_id = f"{prefix}_media_{native_media_id}"
    e.g. 'ig_media_17846368219941196', 'yt_media_Qfus0sZ0s9o'
    """
    media_id:        str
    platform:        str
    parent_media_id: Optional[str] = None   # yt_media_{long_form_id} for Shorts
    is_short:        bool          = False
    format_type:     str           = 'unknown'  # post|reel|video|short|long_form

# ── Heterogeneous Edge schemas ────────────────────────────────────────────────

@dataclass
class EdgePosted:
    """(Author) ──[POSTED]──► (Comment) · all platforms."""
    src_author_id:  str
    dst_comment_id: str
    platform:       str
    timestamp:      str

@dataclass
class EdgeRepliesTo:
    """(Comment) ──[REPLIES_TO]──► (Comment) · Instagram & YouTube only.

    For YouTube, dst is the thread-root comment (threadId), giving a
    flat two-level tree.  Instagram supports arbitrary nesting via parent_id.
    """
    src_comment_id: str
    dst_comment_id: str
    platform:       str

@dataclass
class EdgeBelongsTo:
    """(Comment) ──[BELONGS_TO]──► (Media) · all platforms."""
    src_comment_id: str
    dst_media_id:   str
    platform:       str

@dataclass
class EdgeDerivedFrom:
    """(Media:Short) ──[DERIVED_FROM]──► (Media:LongForm) · YouTube only.

    Populated only when yt_short_to_long is passed to process_batch.
    Build the mapping from Ali/Short_to_Long_connection/ before calling.
    """
    src_media_id: str            # canonical Short   e.g. 'yt_media_b4t6fdYh8ak'
    dst_media_id: str            # canonical LongForm e.g. 'yt_media_gE2Phwx-nKg'
    platform:     str = 'youtube'

@dataclass
class EdgeSimilarTo:
    """(Comment) ──[SIMILAR_TO]──► (Comment) · TikTok only (virtual edge).

    Constructed per-video via cosine similarity on the 15-dim tabular ML
    feature vector as a structural proxy for missing reply chains.
    When LLM topic embeddings (R^K) are available, swap the feature matrix
    in _compute_similar_to_edges() — the interface is identical.
    """
    src_comment_id:   str
    dst_comment_id:   str
    similarity_score: float
    platform:         str = 'tiktok'

# ── Batch container ────────────────────────────────────────────────────────────

@dataclass
class HeteroGraphBatch:
    """All heterogeneous graph components for one or more platform batches.

    Downstream cuGraph usage:
        G = cugraph.from_pandas_edgelist(exporter.to_dataframes(batch)['edges_posted'],
                                          source='src_author_id', destination='dst_comment_id')

    Downstream PyG usage:
        data = HeteroData()
        data['comment'].x = torch.tensor(nodes_comment[feature_cols].values)
        data['comment', 'replies_to', 'comment'].edge_index = ...  # build from edges_replies_to
    """
    platform:           str
    nodes_author:       List[NodeAuthor]      = field(default_factory=list)
    nodes_comment:      List[NodeComment]     = field(default_factory=list)
    nodes_media:        List[NodeMedia]       = field(default_factory=list)
    edges_posted:       List[EdgePosted]      = field(default_factory=list)
    edges_replies_to:   List[EdgeRepliesTo]   = field(default_factory=list)
    edges_belongs_to:   List[EdgeBelongsTo]   = field(default_factory=list)
    edges_derived_from: List[EdgeDerivedFrom] = field(default_factory=list)
    edges_similar_to:   List[EdgeSimilarTo]   = field(default_factory=list)

    def merge(self, other: 'HeteroGraphBatch') -> None:
        """In-place extend of this accumulator with another platform's batch."""
        self.nodes_author.extend(other.nodes_author)
        self.nodes_comment.extend(other.nodes_comment)
        self.nodes_media.extend(other.nodes_media)
        self.edges_posted.extend(other.edges_posted)
        self.edges_replies_to.extend(other.edges_replies_to)
        self.edges_belongs_to.extend(other.edges_belongs_to)
        self.edges_derived_from.extend(other.edges_derived_from)
        self.edges_similar_to.extend(other.edges_similar_to)
        if self.platform != other.platform:
            self.platform = 'multi'

    def summary(self) -> Dict[str, int]:
        return {
            'nodes_author':       len(self.nodes_author),
            'nodes_comment':      len(self.nodes_comment),
            'nodes_media':        len(self.nodes_media),
            'edges_posted':       len(self.edges_posted),
            'edges_replies_to':   len(self.edges_replies_to),
            'edges_belongs_to':   len(self.edges_belongs_to),
            'edges_derived_from': len(self.edges_derived_from),
            'edges_similar_to':   len(self.edges_similar_to),
        }

# ── HeteroGraphExporter ────────────────────────────────────────────────────────

class HeteroGraphExporter:
    """Convert a HeteroGraphBatch to typed DataFrames and write to GCS.

    Explicit dtype specification rationale:
      'string'          – Arrow LargeUtf8; null-safe, compact Parquet encoding
      'category'        – low-cardinality columns (platform, format_type)
                          → Parquet dictionary encoding (~10× smaller)
      'float32'         – embedding placeholder slots (halves storage vs float64)
      'int16'           – topic_vec_dim (max K=32767, well above practical limits)
      pd.BooleanDtype() – nullable boolean (Arrow-native, handles NaN from None)
    """

    # ── Per-table DataFrame builders ─────────────────────────────────────────

    @staticmethod
    def _nodes_author(batch: HeteroGraphBatch) -> pd.DataFrame:
        df = pd.DataFrame([asdict(n) for n in batch.nodes_author])
        if df.empty:
            return pd.DataFrame(columns=['author_id', 'platform'])
        return df.astype({'author_id': 'string', 'platform': 'category'})

    @staticmethod
    def _nodes_comment(batch: HeteroGraphBatch) -> pd.DataFrame:
        _COLS = ['comment_id', 'author_id', 'media_id', 'platform', 'timestamp',
                 'sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'topic_vec_dim']
        df = pd.DataFrame([asdict(n) for n in batch.nodes_comment])
        if df.empty:
            return pd.DataFrame(columns=_COLS)
        return df.astype({
            'comment_id':    'string',
            'author_id':     'string',
            'media_id':      'string',
            'platform':      'category',
            'timestamp':     'string',
            'sentiment_neg': 'float32',
            'sentiment_neu': 'float32',
            'sentiment_pos': 'float32',
            'topic_vec_dim': 'int16',
        })

    @staticmethod
    def _nodes_media(batch: HeteroGraphBatch) -> pd.DataFrame:
        _COLS = ['media_id', 'platform', 'parent_media_id', 'is_short', 'format_type']
        df = pd.DataFrame([asdict(n) for n in batch.nodes_media])
        if df.empty:
            return pd.DataFrame(columns=_COLS)
        df = df.astype({'media_id': 'string', 'platform': 'category', 'format_type': 'category'})
        df['parent_media_id'] = df['parent_media_id'].astype('string')
        df['is_short']        = df['is_short'].astype(pd.BooleanDtype())
        return df

    @staticmethod
    def _edges_posted(batch: HeteroGraphBatch) -> pd.DataFrame:
        df = pd.DataFrame([asdict(e) for e in batch.edges_posted])
        if df.empty:
            return pd.DataFrame(columns=['src_author_id', 'dst_comment_id', 'platform', 'timestamp'])
        return df.astype({'src_author_id': 'string', 'dst_comment_id': 'string',
                          'platform': 'category', 'timestamp': 'string'})

    @staticmethod
    def _edges_replies_to(batch: HeteroGraphBatch) -> pd.DataFrame:
        df = pd.DataFrame([asdict(e) for e in batch.edges_replies_to])
        if df.empty:
            return pd.DataFrame(columns=['src_comment_id', 'dst_comment_id', 'platform'])
        return df.astype({'src_comment_id': 'string', 'dst_comment_id': 'string',
                          'platform': 'category'})

    @staticmethod
    def _edges_belongs_to(batch: HeteroGraphBatch) -> pd.DataFrame:
        df = pd.DataFrame([asdict(e) for e in batch.edges_belongs_to])
        if df.empty:
            return pd.DataFrame(columns=['src_comment_id', 'dst_media_id', 'platform'])
        return df.astype({'src_comment_id': 'string', 'dst_media_id': 'string',
                          'platform': 'category'})

    @staticmethod
    def _edges_derived_from(batch: HeteroGraphBatch) -> pd.DataFrame:
        df = pd.DataFrame([asdict(e) for e in batch.edges_derived_from])
        if df.empty:
            return pd.DataFrame(columns=['src_media_id', 'dst_media_id', 'platform'])
        return df.astype({'src_media_id': 'string', 'dst_media_id': 'string',
                          'platform': 'category'})

    @staticmethod
    def _edges_similar_to(batch: HeteroGraphBatch) -> pd.DataFrame:
        df = pd.DataFrame([asdict(e) for e in batch.edges_similar_to])
        if df.empty:
            return pd.DataFrame(columns=['src_comment_id', 'dst_comment_id',
                                         'similarity_score', 'platform'])
        return df.astype({'src_comment_id': 'string', 'dst_comment_id': 'string',
                          'similarity_score': 'float32', 'platform': 'category'})

    def to_dataframes(self, batch: HeteroGraphBatch) -> Dict[str, pd.DataFrame]:
        """Return {parquet_stem → typed DataFrame} for all 8 output files."""
        return {
            'nodes_author':       self._nodes_author(batch),
            'nodes_comment':      self._nodes_comment(batch),
            'nodes_media':        self._nodes_media(batch),
            'edges_posted':       self._edges_posted(batch),
            'edges_replies_to':   self._edges_replies_to(batch),
            'edges_belongs_to':   self._edges_belongs_to(batch),
            'edges_derived_from': self._edges_derived_from(batch),
            'edges_similar_to':   self._edges_similar_to(batch),
        }

    def export_to_gcs(
        self,
        batch:      HeteroGraphBatch,
        output_gcs: str,
        fs:         Any,             # gcsfs.GCSFileSystem
    ) -> Dict[str, int]:
        """Write all 8 Parquet files to GCS; return {stem: row_count} manifest."""
        frames  = self.to_dataframes(batch)
        counts: Dict[str, int] = {}
        base    = output_gcs.rstrip('/')
        for name, df in frames.items():
            uri = f'{base}/{name}.parquet'
            df.to_parquet(uri, compression='snappy', index=False)
            counts[name] = len(df)
            logger.info('Written %8d rows → %s', len(df), uri)
        return counts

# ── Module-level native-ID extractors (one per platform) ─────────────────────

def _native_id_instagram(record: Dict[str, Any]) -> str:
    return str(record.get('comment_id', record.get('id', '')))

def _native_id_tiktok(record: Dict[str, Any]) -> str:
    for field in ('cid', 'comment_id', 'id'):
        val = record.get(field)
        if val is not None and str(val).strip():
            return str(val)
    return ''

def _native_id_youtube(record: Dict[str, Any]) -> str:
    return str(record.get('commentId', record.get('id', '')))

# ── UnifiedPipeline ────────────────────────────────────────────────────────────

class UnifiedPipeline:
    """Production-grade cross-platform data preparation pipeline.

    Returns three parallel output streams from process_batch():
        ml_results   : List[ProcessedCommentML]         — tabular features (unchanged)
        hetero_batch : HeteroGraphBatch                 — heterogeneous graph
        llm_results  : List[Tuple[str, RawComment]]     — raw text (unchanged)
    """

    def __init__(self, output_dir: Path = Path('./output')) -> None:
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        self.processed_count: int = 0
        self.error_count:     int = 0
        self.errors: List[Dict[str, Any]] = []

    # ── Canonical ID builders ─────────────────────────────────────────────────

    @staticmethod
    def _pfx(platform: str) -> str:
        return PLATFORM_PREFIX.get(platform, platform[:2])

    @staticmethod
    def _canon_author(platform: str, native_uid: str) -> str:
        pfx = PLATFORM_PREFIX.get(platform, platform[:2])
        return f'{pfx}_author_{native_uid}'

    @staticmethod
    def _canon_comment(platform: str, native_cid: str) -> str:
        pfx = PLATFORM_PREFIX.get(platform, platform[:2])
        return f'{pfx}_comment_{native_cid}'

    @staticmethod
    def _canon_media(platform: str, native_media_id: str) -> str:
        pfx = PLATFORM_PREFIX.get(platform, platform[:2])
        return f'{pfx}_media_{native_media_id}'

    # ── Platform normalisation ────────────────────────────────────────────────

    def normalize_raw_comment(
        self,
        record:   Dict[str, Any],
        platform: str,
    ) -> Optional[Tuple[RawComment, str]]:
        """Normalise a raw record to (RawComment, native_comment_id).

        native_comment_id is the un-prefixed platform ID; the caller namespaces
        it via _canon_comment().  reply_to_comment_id in RawComment stores the
        raw native parent ID (un-prefixed); Pass 2 of process_batch resolves it.
        Returns None for structurally invalid records.
        """
        try:
            p = platform.lower()

            if p == 'instagram':
                native_cid    = _native_id_instagram(record)
                author_id     = str(record.get('from_id', ''))
                media_id      = str(record.get('media_id', ''))
                comment_text  = str(record.get('text', ''))
                raw_parent    = record.get('parent_id')
                native_reply: Optional[str] = str(raw_parent) if raw_parent else None
                timestamp     = str(record.get('timestamp', ''))

            elif p == 'tiktok':
                native_cid   = _native_id_tiktok(record)
                author_id    = str(record.get('uid', ''))
                media_id     = str(record.get('media_id', record.get('video_id', '')))
                comment_text = str(record.get('text', ''))
                native_reply = None          # dataset is structurally flat
                raw_ts       = record.get('create_time', record.get('timestamp', ''))
                if isinstance(raw_ts, (int, float)) and raw_ts > 0:
                    timestamp = datetime.fromtimestamp(raw_ts, tz=timezone.utc).isoformat()
                else:
                    timestamp = str(raw_ts) if raw_ts else ''

            elif p == 'youtube':
                native_cid   = _native_id_youtube(record)
                author_id    = str(record.get('authorChannelId', ''))
                media_id     = str(record.get('videoId', ''))
                comment_text = str(record.get('textOriginal', ''))
                is_reply     = bool(record.get('isReply', False))
                thread_id    = record.get('threadId', '')
                # threadId == commentId for root comments; only use when isReply=True
                native_reply = str(thread_id) if (is_reply and thread_id) else None
                raw_ts       = record.get('publishedAt', record.get('updatedAt', ''))
                timestamp    = str(raw_ts) if raw_ts else ''

            else:
                raise ValueError(f'Unknown platform: {platform!r}')

            if not native_cid:
                logger.debug('[%s] Skipping — empty native comment ID', p)
                return None
            if not author_id:
                logger.debug('[%s] Skipping %r — missing author_id', p, native_cid)
                return None
            if not media_id:
                logger.debug('[%s] Skipping %r — missing media_id', p, native_cid)
                return None

            raw = RawComment(
                platform=p,
                author_id=author_id,
                media_id=media_id,
                comment_text=comment_text,
                reply_to_comment_id=native_reply,  # raw parent ID, not yet namespaced
                timestamp=timestamp,
                original_from_id=record.get('from_id'),
                original_uid=record.get('uid'),
            )
            return raw, native_cid

        except Exception as exc:
            logger.error('[%s] normalize_raw_comment failed: %s', platform, exc)
            self.errors.append({'platform': platform, 'error': str(exc)})
            return None

    # ── ML feature extraction (UNCHANGED) ─────────────────────────────────────

    def process_comment(
        self,
        raw:        RawComment,
        comment_id: str,
    ) -> Optional[ProcessedCommentML]:
        """Derive tabular ML features from a normalised RawComment.

        GML graph construction is handled separately in _build_hetero_batch()
        so the two concerns remain fully decoupled.
        """
        try:
            features = advanced_text_preprocessing(raw.comment_text)
            return ProcessedCommentML(
                comment_id=comment_id,
                author_id=raw.author_id,
                media_id=raw.media_id,
                platform=raw.platform,
                text_length=features['text_length'],
                word_count=features['word_count'],
                emoji_count=features['emoji_count'],
                unique_emoji_count=float(features['unique_emoji_count']),
                emoji_entropy=features['emoji_entropy'],
                emoji_variety_ratio=features['emoji_variety_ratio'],
                emoji_per_word_ratio=features['emoji_per_word_ratio'],
                url_count=features['url_count'],
                mention_count=features['mention_count'],
                hashtag_count=features['hashtag_count'],
                exclamation_count=features['exclamation_count'],
                question_count=features['question_count'],
                avg_word_length=features['avg_word_length'],
                has_numbers=features['has_numbers'],
                has_links=features['has_links'],
                timestamp=raw.timestamp,
            )
        except Exception as exc:
            logger.error('process_comment failed [%s]: %s', comment_id, exc)
            self.errors.append({'comment_id': comment_id, 'error': str(exc)})
            return None

    # ── TikTok virtual edge computation ──────────────────────────────────────

    @staticmethod
    def _compute_similar_to_edges(
        comment_ids:          List[str],
        feature_matrix:       np.ndarray,    # shape (n, d), any float dtype
        platform:             str   = 'tiktok',
        similarity_threshold: float = 0.85,
        max_neighbors:        int   = 5,
    ) -> List[EdgeSimilarTo]:
        """Build SIMILAR_TO virtual edges via cosine similarity.

        Rows of feature_matrix are L2-normalised; the Gram matrix dot-product
        then equals pairwise cosine similarity.  Self-loops are suppressed.
        Each source node gets at most max_neighbors outgoing edges (top-k).

        Invoke per media_id group (see _build_hetero_batch) to bound complexity
        at O(n_per_video²) rather than O(n_platform²).

        Interface note: replace feature_matrix with an (n, K) LLM topic
        embedding array when R^K vectors become available — no other changes.
        """
        n = len(comment_ids)
        if n < 2 or feature_matrix.shape[0] != n:
            return []

        F = feature_matrix.astype(np.float32)
        norms = np.linalg.norm(F, axis=1, keepdims=True)
        norms = np.where(norms == 0.0, 1.0, norms)
        F /= norms                                  # in-place unit-row normalisation

        S = F @ F.T                                  # (n, n) cosine similarity matrix
        np.fill_diagonal(S, -1.0)                   # suppress self-loops

        edges: List[EdgeSimilarTo] = []
        for i in range(n):
            above = np.where(S[i] >= similarity_threshold)[0]
            if above.size == 0:
                continue
            top_k = above[np.argsort(S[i, above])[::-1][:max_neighbors]]
            for j in top_k:
                edges.append(EdgeSimilarTo(
                    src_comment_id=comment_ids[i],
                    dst_comment_id=comment_ids[int(j)],
                    similarity_score=float(S[i, j]),
                    platform=platform,
                ))
        return edges

    # ── Heterogeneous graph construction ──────────────────────────────────────

    def _build_hetero_batch(
        self,
        resolved_records:     List[Tuple[RawComment, str, Optional[str]]],
        ml_results:           List[ProcessedCommentML],
        platform:             str,
        yt_short_to_long:     Optional[Dict[str, str]] = None,
        tiktok_sim_threshold: float = 0.85,
        tiktok_max_neighbors: int   = 5,
    ) -> HeteroGraphBatch:
        """Build all heterogeneous nodes and edges for one resolved platform batch.

        resolved_records: (raw, canonical_comment_id, resolved_reply_canon_id | None)
            — same length and order as ml_results (errors already filtered out).
        yt_short_to_long: {canon_short_media_id → canon_long_media_id}
            — pre-built from Ali/Short_to_Long_connection/; YouTube only.
        """
        p     = platform.lower()
        batch = HeteroGraphBatch(platform=p)

        seen_authors: set = set()
        seen_media:   set = set()
        ml_by_cid: Dict[str, ProcessedCommentML] = {m.comment_id: m for m in ml_results}

        # ── Node pass ─────────────────────────────────────────────────────────
        for raw, canon_cid, _ in resolved_records:
            canon_author = self._canon_author(p, raw.author_id)
            canon_media  = self._canon_media(p, raw.media_id)

            # Author node — deduplicated within batch
            if canon_author not in seen_authors:
                batch.nodes_author.append(NodeAuthor(author_id=canon_author, platform=p))
                seen_authors.add(canon_author)

            # Media node — deduplicated; platform-specific metadata injected here
            if canon_media not in seen_media:
                is_short:  bool          = False
                parent_id: Optional[str] = None

                if p == 'youtube':
                    fmt = 'long_form'
                    if yt_short_to_long and canon_media in yt_short_to_long:
                        is_short  = True
                        parent_id = yt_short_to_long[canon_media]
                        fmt       = 'short'
                elif p == 'instagram':
                    fmt = 'post'
                elif p == 'tiktok':
                    fmt = 'video'
                else:
                    fmt = 'unknown'

                batch.nodes_media.append(NodeMedia(
                    media_id=canon_media, platform=p,
                    parent_media_id=parent_id, is_short=is_short, format_type=fmt,
                ))
                seen_media.add(canon_media)

            # Comment node — embedding slots pre-zeroed (filled in downstream)
            batch.nodes_comment.append(NodeComment(
                comment_id=canon_cid,
                author_id=canon_author,
                media_id=canon_media,
                platform=p,
                timestamp=raw.timestamp,
            ))

        # ── Edge pass ─────────────────────────────────────────────────────────
        for raw, canon_cid, resolved_reply_id in resolved_records:
            canon_author = self._canon_author(p, raw.author_id)
            canon_media  = self._canon_media(p, raw.media_id)

            batch.edges_posted.append(EdgePosted(
                src_author_id=canon_author,
                dst_comment_id=canon_cid,
                platform=p,
                timestamp=raw.timestamp,
            ))
            batch.edges_belongs_to.append(EdgeBelongsTo(
                src_comment_id=canon_cid,
                dst_media_id=canon_media,
                platform=p,
            ))
            if resolved_reply_id is not None:
                batch.edges_replies_to.append(EdgeRepliesTo(
                    src_comment_id=canon_cid,
                    dst_comment_id=resolved_reply_id,
                    platform=p,
                ))

        # ── DERIVED_FROM edges (YouTube Shorts → LongForm) ────────────────────
        if p == 'youtube' and yt_short_to_long:
            for short_mid, long_mid in yt_short_to_long.items():
                if short_mid in seen_media:        # only emit if Short has comments
                    batch.edges_derived_from.append(
                        EdgeDerivedFrom(src_media_id=short_mid, dst_media_id=long_mid)
                    )
            shorts_in_batch = sum(1 for m in batch.nodes_media if m.is_short)
            logger.info(
                '[%s] DERIVED_FROM: %d Short→LongForm edges (Shorts with comments: %d)',
                p, len(batch.edges_derived_from), shorts_in_batch,
            )

        # ── SIMILAR_TO edges (TikTok — per-video cosine similarity) ──────────
        if p == 'tiktok' and len(resolved_records) >= 2:
            from collections import defaultdict as _dd
            media_groups: Dict[str, List[Tuple[str, ProcessedCommentML]]] = _dd(list)
            for raw, canon_cid, _ in resolved_records:
                ml = ml_by_cid.get(canon_cid)
                if ml is not None:
                    media_groups[raw.media_id].append((canon_cid, ml))

            total_sim_edges = 0
            for _video_id, group in media_groups.items():
                if len(group) < 2:
                    continue
                cids = [cid for cid, _ in group]
                feat = np.array(
                    [[getattr(ml, f, 0.0) for f in _TIKTOK_SIM_FEATURES] for _, ml in group],
                    dtype=np.float32,
                )
                sim_edges = self._compute_similar_to_edges(
                    cids, feat, p, tiktok_sim_threshold, tiktok_max_neighbors,
                )
                batch.edges_similar_to.extend(sim_edges)
                total_sim_edges += len(sim_edges)

            logger.info(
                '[%s] SIMILAR_TO: %d virtual edges across %d videos (threshold=%.2f, k=%d)',
                p, total_sim_edges, len(media_groups),
                tiktok_sim_threshold, tiktok_max_neighbors,
            )

        logger.info('[%s] HeteroGraphBatch summary: %s', p, batch.summary())
        return batch

    # ── Batch orchestration ───────────────────────────────────────────────────

    def process_batch(
        self,
        raw_records:          List[Dict[str, Any]],
        platform:             str,
        yt_short_to_long:     Optional[Dict[str, str]] = None,
        tiktok_sim_threshold: float = 0.85,
        tiktok_max_neighbors: int   = 5,
    ) -> Tuple[List[ProcessedCommentML], HeteroGraphBatch, List[Tuple[str, RawComment]]]:
        """Two-pass batch processor: ML features + hetero graph + LLM tuples.

        Pass 1 — Normalisation & node-universe registration
            normalize_raw_comment() → (RawComment, native_cid).
            canonical_cid = _canon_comment(platform, native_cid) registered in
            node_universe.  Duplicate native_cids keep first occurrence only.

        Pass 2 — ML extraction + orphan-pruned edge resolution
            • ML: process_comment() → ProcessedCommentML.
            • GML: raw.reply_to_comment_id namespaced via _canon_comment() and
              validated against node_universe.  Absent parents → pruned to None,
              preventing out-of-bounds in PyG edge_index loaders.
            • LLM: (canonical_cid, RawComment) tuple collected.
            Resolved tuples forwarded to _build_hetero_batch() for graph assembly.

        Returns
        -------
        ml_results   : tabular ML records (unchanged format and semantics)
        hetero_batch : HeteroGraphBatch with all node and edge lists
        llm_results  : (comment_id, RawComment) tuples for JSONL export
        """
        p = platform.lower()
        ml_results:  List[ProcessedCommentML]     = []
        llm_results: List[Tuple[str, RawComment]] = []

        # ── Pass 1 ────────────────────────────────────────────────────────────
        normalised: List[Tuple[RawComment, str, str]] = []   # (raw, native_cid, canon_cid)
        node_universe: set = set()

        for idx, record in enumerate(raw_records):
            result = self.normalize_raw_comment(record, p)
            if result is None:
                self.error_count += 1
                continue
            raw, native_cid = result
            canon_cid = self._canon_comment(p, native_cid)
            if canon_cid in node_universe:
                logger.debug('[%s] Duplicate native_cid %r at index %d — skipped', p, native_cid, idx)
                continue
            node_universe.add(canon_cid)
            normalised.append((raw, native_cid, canon_cid))

        logger.info('[%s] Pass 1: %d valid / %d skipped (batch=%d)',
                    p, len(normalised), len(raw_records) - len(normalised), len(raw_records))

        # ── Pass 2 ────────────────────────────────────────────────────────────
        resolved_records: List[Tuple[RawComment, str, Optional[str]]] = []
        orphans_pruned = 0

        for raw, _native_cid, canon_cid in normalised:
            resolved_reply: Optional[str] = None
            if raw.reply_to_comment_id is not None:
                candidate = self._canon_comment(p, raw.reply_to_comment_id)
                if candidate in node_universe:
                    resolved_reply = candidate
                else:
                    orphans_pruned += 1

            ml = self.process_comment(raw, canon_cid)
            if ml is not None:
                ml_results.append(ml)
                llm_results.append((canon_cid, raw))
                resolved_records.append((raw, canon_cid, resolved_reply))
                self.processed_count += 1
            else:
                self.error_count += 1

        reply_count = sum(1 for _, _, r in resolved_records if r is not None)
        logger.info('[%s] Pass 2: %d processed | %d reply edges | %d orphans pruned',
                    p, len(ml_results), reply_count, orphans_pruned)

        # ── GML: assemble heterogeneous graph ─────────────────────────────────
        hetero_batch = self._build_hetero_batch(
            resolved_records=resolved_records,
            ml_results=ml_results,
            platform=p,
            yt_short_to_long=yt_short_to_long,
            tiktok_sim_threshold=tiktok_sim_threshold,
            tiktok_max_neighbors=tiktok_max_neighbors,
        )

        return ml_results, hetero_batch, llm_results


## Section 6: Testing with Mock Data (Instagram, Facebook, TikTok)

In [ ]:

def test_pipeline_with_mock_data():
    """Sanity-check the hybrid hetero-graph pipeline against synthetic records.

    Verifies all five edge types, orphan pruning, TikTok flatness,
    YouTube DERIVED_FROM injection, and SIMILAR_TO virtual edge creation.
    """
    pipeline = UnifiedPipeline(output_dir=Path('/tmp/test_output'))

    # Fake Short→LongForm mapping (canonical IDs already)
    yt_short_to_long = {
        'yt_media_SHORT_v01': 'yt_media_LONG_v99',
    }

    mock = {
        'instagram': [
            {'comment_id': 'c001', 'from_id': 'u1', 'media_id': 'p101',
             'text': 'Love this! ❤️😍 #awesome', 'parent_id': None,
             'timestamp': '2026-03-01T10:30:00Z'},
            {'comment_id': 'c002', 'from_id': 'u2', 'media_id': 'p101',
             'text': 'Fire content 🔥 @u1', 'parent_id': 'c001',  # → ig_comment_c001
             'timestamp': '2026-03-01T10:35:00Z'},
            {'comment_id': 'c003', 'from_id': 'u3', 'media_id': 'p101',
             'text': 'Great stuff!', 'parent_id': 'c999',          # orphan → pruned
             'timestamp': '2026-03-01T10:40:00Z'},
        ],
        'tiktok': [
            {'cid': 'tk001', 'uid': 'tk_u1', 'media_id': 'v301',
             'text': 'So good! 🥳✨ #viral', 'reply_id': 0, 'create_time': 1711080000},
            {'cid': 'tk002', 'uid': 'tk_u2', 'media_id': 'v301',
             'text': 'Best ever! 🌟💯', 'reply_id': 'tk001',  # junk → forced None
             'create_time': 1711080300},
            {'cid': 'tk003', 'uid': 'tk_u3', 'media_id': 'v301',
             'text': 'Absolutely amazing!!! ❤️', 'reply_id': 0, 'create_time': 1711080600},
        ],
        'youtube': [
            # Root comment on a Short video (matched in yt_short_to_long)
            {'commentId': 'yt_c001', 'authorChannelId': 'yt_u1', 'videoId': 'SHORT_v01',
             'textOriginal': 'Short is 🔥', 'isReply': False, 'threadId': 'yt_c001',
             'publishedAt': '2026-03-05T12:00:00Z'},
            # Reply to yt_c001 — should resolve to yt_comment_yt_c001
            {'commentId': 'yt_c002', 'authorChannelId': 'yt_u2', 'videoId': 'SHORT_v01',
             'textOriginal': 'Totally agree!', 'isReply': True, 'threadId': 'yt_c001',
             'publishedAt': '2026-03-05T12:05:00Z'},
            # Root comment on a LongForm video
            {'commentId': 'yt_c003', 'authorChannelId': 'yt_u3', 'videoId': 'LONG_v99',
             'textOriginal': 'Great long video 👍', 'isReply': False, 'threadId': 'yt_c003',
             'publishedAt': '2026-03-05T13:00:00Z'},
        ],
    }

    print('=' * 70)
    all_ml: List[ProcessedCommentML] = []
    graph = HeteroGraphBatch(platform='multi')

    for platform, records in mock.items():
        kwargs = {}
        if platform == 'youtube':
            kwargs['yt_short_to_long'] = yt_short_to_long
        if platform == 'tiktok':
            kwargs['tiktok_sim_threshold'] = 0.0   # force all pairs to get edges
            kwargs['tiktok_max_neighbors']  = 2

        ml, batch, llm = pipeline.process_batch(records, platform, **kwargs)
        all_ml.extend(ml)
        graph.merge(batch)

        print(f'\n[{platform}]  {len(ml)} ML / {len(llm)} LLM records')
        print(f'  graph: {batch.summary()}')
        for g in batch.edges_replies_to:
            print(f'  REPLIES_TO: {g.src_comment_id} → {g.dst_comment_id}')
        for g in batch.edges_derived_from:
            print(f'  DERIVED_FROM: {g.src_media_id} → {g.dst_media_id}')
        for g in batch.edges_similar_to[:3]:
            print(f'  SIMILAR_TO: {g.src_comment_id} → {g.dst_comment_id}  sim={g.similarity_score:.3f}')

    print(f'\nTotal processed: {pipeline.processed_count}  |  errors: {pipeline.error_count}')
    print(f'Merged graph: {graph.summary()}')

    # ── Assertions ────────────────────────────────────────────────────────────
    c_by_id = {n.comment_id: n for n in graph.nodes_comment}
    e_replies = {(e.src_comment_id, e.dst_comment_id) for e in graph.edges_replies_to}
    e_derived = {(e.src_media_id, e.dst_media_id)     for e in graph.edges_derived_from}
    m_by_id   = {m.media_id: m                        for m in graph.nodes_media}

    # Canonical ID format
    assert 'ig_comment_c001' in c_by_id,            'IG root node missing'
    assert 'tk_comment_tk001' in c_by_id,           'TikTok node missing'
    assert 'yt_comment_yt_c001' in c_by_id,         'YT root node missing'

    # Instagram reply edge resolved correctly
    assert ('ig_comment_c002', 'ig_comment_c001') in e_replies, \
        'IG REPLIES_TO edge not resolved'

    # Orphan edge pruned (c999 not in batch)
    assert all(e[1] != 'ig_comment_c999' for e in e_replies), \
        'Orphan edge not pruned'

    # TikTok: no REPLIES_TO edges (all forced to None)
    tk_replies = [e for e in graph.edges_replies_to if e.platform == 'tiktok']
    assert len(tk_replies) == 0, f'TikTok reply edges present: {tk_replies}'

    # TikTok: SIMILAR_TO edges generated
    assert len(graph.edges_similar_to) > 0, 'No TikTok SIMILAR_TO edges generated'
    assert all(e.platform == 'tiktok' for e in graph.edges_similar_to), \
        'SIMILAR_TO edges on wrong platform'

    # YouTube REPLIES_TO resolved
    assert ('yt_comment_yt_c002', 'yt_comment_yt_c001') in e_replies, \
        'YT reply not resolved'

    # YouTube DERIVED_FROM (Short → LongForm)
    assert ('yt_media_SHORT_v01', 'yt_media_LONG_v99') in e_derived, \
        'DERIVED_FROM edge missing'

    # Short media node flagged correctly
    assert m_by_id['yt_media_SHORT_v01'].is_short is True,   'Short not flagged'
    assert m_by_id['yt_media_SHORT_v01'].format_type == 'short', 'format_type wrong'
    assert m_by_id['yt_media_LONG_v99'].is_short is False,   'LongForm wrongly flagged'

    # LongForm media exists as node (from yt_c003's videoId)
    assert 'yt_media_LONG_v99' in m_by_id, 'LongForm media node missing'

    # HeteroGraphExporter produces typed DataFrames
    exporter = HeteroGraphExporter()
    frames   = exporter.to_dataframes(graph)
    assert set(frames.keys()) == {
        'nodes_author', 'nodes_comment', 'nodes_media',
        'edges_posted', 'edges_replies_to', 'edges_belongs_to',
        'edges_derived_from', 'edges_similar_to',
    }, 'Missing output files'
    assert frames['nodes_comment']['sentiment_neg'].dtype == np.float32, 'Wrong dtype'
    assert str(frames['nodes_comment']['platform'].dtype) == 'category', 'Wrong dtype'

    print('\n✓ All assertions passed — Hybrid GML pipeline verified.')
    print('=' * 70)

test_pipeline_with_mock_data()


In [ ]:
import gcsfs

fs  = gcsfs.GCSFileSystem(project=GCP_PROJECT_ID)
dfs: Dict[str, pd.DataFrame] = {}

# Instagram
try:
    dfs['instagram'] = pd.read_parquet(f"{gcs}/ig_comments_cleaned.parquet")
    logger.info("instagram : %9d rows", len(dfs['instagram']))
except Exception as e:
    logger.error("instagram  FAILED: %s", e)

# TikTok
try:
    dfs['tiktok'] = pd.read_parquet(f"{gcs}/tk_comments_clean.parquet")
    logger.info("tiktok    : %9d rows", len(dfs['tiktok']))
except Exception as e:
    logger.error("tiktok     FAILED: %s", e)

# YouTube (4 part-files)
try:
    yt_parts = [pd.read_parquet(f"{gcs}/YTcomments_{i}_cleaned.parquet") for i in range(1, 5)]
    dfs['youtube'] = pd.concat(yt_parts, ignore_index=True)
    logger.info("youtube   : %9d rows  (4 files)", len(dfs['youtube']))
except Exception as e:
    logger.error("youtube    FAILED: %s", e)

logger.info("Loaded platforms: %s", list(dfs.keys()))

In [ ]:
# Sample run — 500 rows per platform to verify normalisation before the full production run
sample_size    = 500
pipeline_sample = UnifiedPipeline(output_dir=Path('/tmp/sample_output'))
all_ml_s:  List[ProcessedCommentML]     = []
all_llm_s: List[Tuple[str, RawComment]] = []

for platform in ['instagram', 'tiktok', 'youtube']:
    df = dfs.get(platform)
    if df is None:
        logger.warning("  %s: not loaded, skipping", platform)
        continue
    sample = df.head(sample_size).to_dict('records')
    ml, batch, llm = pipeline_sample.process_batch(sample, platform)
    all_ml_s.extend(ml)
    all_llm_s.extend(llm)
    logger.info("  %s: %d ML / %d LLM records | graph: %s", platform, len(ml), len(llm), batch.summary())

logger.info("Sample total: %d  |  errors: %d",
            pipeline_sample.processed_count, pipeline_sample.error_count)

# Quick spot-check: log first 2 LLM records
for cid, raw in all_llm_s[:2]:
    logger.info(json.dumps(
        {"comment_id": cid, "text": raw.comment_text[:60], "platform": raw.platform},
        ensure_ascii=False,
    ))

In [ ]:

import traceback

output_preped = f'{gcs}/Preped_Comments'   # ML + LLM outputs (unchanged)
output_hetero = f'{gcs}/HeteroGraph'       # NEW: 8 hetero-graph Parquet files

# ── YouTube Short connection mapping ──────────────────────────────────────────
# Source: Ali/Short_to_Long_connection/shorts_connection_mapping_2021-2026_full.csv
#         (895 connected pairs: 797 Short→LongForm, 98 Short→AnotherShort)
# Upload to GCS before running:
#   gsutil cp "Ali/Short_to_Long_connection/shorts_connection_mapping_2021-2026_full.csv" \
#             gs://afb_showreel/
_MAPPING_URI = f'{gcs}/shorts_connection_mapping_2021-2026_full.csv'
try:
    with fs.open(_MAPPING_URI) as _f:
        _mapping_df = pd.read_csv(_f)
    _connected = _mapping_df[_mapping_df['status'] == 'connected']
    yt_short_to_long: Dict[str, str] = {
        f'yt_media_{row["short_id"]}': f'yt_media_{row["connected_id"]}'
        for _, row in _connected.iterrows()
    }
    _long_n  = (_connected['connection_type'] == 'long').sum()
    _short_n = (_connected['connection_type'] == 'short').sum()
    logger.info('Short mapping loaded: %d pairs (%d Short→Long, %d Short→Short)',
                len(yt_short_to_long), _long_n, _short_n)
except Exception as _e:
    logger.warning('Short mapping not found in GCS — DERIVED_FROM edges skipped. (%s)', _e)
    yt_short_to_long = {}

# ── Main pipeline (wrapped so scheduler marks run FAILED on any exception) ────
try:
    pipeline     = UnifiedPipeline(output_dir=Path('/tmp/preped_output'))
    all_ml:  List[ProcessedCommentML]     = []
    all_llm: List[Tuple[str, RawComment]] = []
    hetero_graph = HeteroGraphBatch(platform='multi')

    for platform in ['instagram', 'tiktok', 'youtube']:
        df = dfs.get(platform)
        if df is None:
            logger.warning('Skipping %s — not loaded', platform)
            continue
        logger.info('Processing %s  (%d rows)…', platform, len(df))

        kwargs: Dict[str, Any] = {}
        if platform == 'youtube':
            kwargs['yt_short_to_long'] = yt_short_to_long

        ml, batch, llm = pipeline.process_batch(df.to_dict('records'), platform, **kwargs)
        all_ml.extend(ml)
        all_llm.extend(llm)
        hetero_graph.merge(batch)
        logger.info('  → %d records OK', len(ml))

    logger.info('Total processed: %d  |  errors: %d',
                pipeline.processed_count, pipeline.error_count)

    # ── Write ML + LLM outputs (UNCHANGED) ───────────────────────────────────
    llm_uri = f'{output_preped}/comments_llm.jsonl'
    with fs.open(llm_uri, 'w', encoding='utf-8') as fh:
        for cid, raw in all_llm:
            fh.write(json.dumps({
                'comment_id': cid,
                'text':       raw.comment_text,
                'author_id':  raw.author_id,
                'platform':   raw.platform,
                'timestamp':  raw.timestamp,
            }, ensure_ascii=False) + '\n')
    logger.info('Written %d lines → %s', len(all_llm), llm_uri)

    ml_df = pd.DataFrame([asdict(r) for r in all_ml])
    ml_df.to_parquet(f'{output_preped}/comments_ml.parquet', compression='snappy', index=False)
    logger.info('Written %d rows → %s/comments_ml.parquet', len(ml_df), output_preped)

    # ── Write heterogeneous graph outputs (NEW) ───────────────────────────────
    exporter = HeteroGraphExporter()
    counts   = exporter.export_to_gcs(hetero_graph, output_hetero, fs)

    logger.info('\n=== HeteroGraph export  (gs://%s/HeteroGraph/) ===', GCP_BUCKET)
    for name, n in counts.items():
        logger.info('  %-28s %8d rows', name, n)

    logger.info('\n=== ML platform distribution ===')
    logger.info('\n' + ml_df['platform'].value_counts().to_string())

    logger.info('\n=== HeteroGraph node / edge counts ===')
    for k, v in hetero_graph.summary().items():
        logger.info('  %-28s %8d', k, v)

    logger.info('Pipeline finished OK')

except Exception:
    logger.error('FATAL — pipeline aborted:\n%s', traceback.format_exc())
    raise   # re-raise so Colab Enterprise marks the scheduled run as FAILED
